# Demo 06a — Decision Trees

**Class 06 · Blocks 1-2** (slides: `slides/06_trees_knn.md`)

Five live demos, each with more experiments than the slides show:

* **T1 — Impurity and the best split:** entropy and Gini by hand on a 14-row table, then on real data.
* **T2 — A tree from scratch:** the greedy algorithm in 40 lines of NumPy, against scikit-learn.
* **T3 — Depth, overfitting and pruning:** validation curves, cost-complexity pruning, the rules of a small tree.
* **T4 — Regression trees:** variance reduction, the staircase, the failure outside the data.
* **T5 — Instability and importances:** many trees from many samples, and why importances can lie.

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
DATA = Path("../../datasets")


def table(rows, cols):
    """Print a list of rows as a polars table (hyperparameters are written with `:g`)."""
    keep = {"alpha", "ccp_alpha", "max_depth", "min_leaf", "k"}
    rows = [tuple(f"{v:g}" if c in keep and isinstance(v, float) else v for c, v in zip(cols, r)) for r in rows]
    print(pl.DataFrame(rows, schema=cols, orient="row"))


def load_table(name, target):
    """Read `datasets/<name>.csv.zst` with polars: null -> median / "missing", text -> one-hot."""
    df = pl.read_csv(DATA / f"{name}.csv.zst")
    X = df.drop(target)
    X = X.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c, t in X.schema.items() if t.is_numeric()])
    cats = [c for c, t in X.schema.items() if t == pl.String]
    if cats:
        X = X.with_columns([pl.col(c).fill_null("missing") for c in cats]).to_dummies(columns=cats)
    return X.to_numpy().astype(float), df[target].to_numpy(), X.columns


def load_california():
    """California housing with the logarithm of the four skewed features (Class 03)."""
    X, y, names = load_table("california_housing", "MedHouseVal")
    for j, n in enumerate(names):
        if n in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
            X[:, j] = np.log(X[:, j])
    return X, y.astype(float), names


Xb, yb, names_b = load_table("breast_cancer", "target")
Xg, cls_g, names_g = load_table("german_credit", "class")
yg = (cls_g == "bad").astype(int)
Xh, num_h, names_h = load_table("heart_disease", "num")
yh = (num_h == ">50_1").astype(int)
Xw, q_w, names_w = load_table("wine_quality", "quality")
yw = (q_w >= 6).astype(int)
Xd, yd, names_d = load_table("digits", "target")
CLASSIFICATION = {"breast_cancer": (Xb, yb), "german_credit": (Xg, yg), "heart_disease": (Xh, yh), "wine (quality >= 6)": (Xw, yw), "digits": (Xd, yd.astype(int))}
for k, (X_, y_) in CLASSIFICATION.items():
    print(f"{k:<22} {X_.shape[0]:>5} rows x {X_.shape[1]:>3} features, classes {np.bincount(y_).tolist()}")

## T1 — Impurity and the best split

A tree asks a sequence of yes/no questions. Each question should make the groups it creates **purer**: more
examples of one class. The **impurity** of a group with class proportions $p_1,\dots,p_K$ is

* **entropy** $H = -\sum_k p_k \log_2 p_k$ (the bits needed to name the class, Class 01),
* **Gini** $G = 1 - \sum_k p_k^2$ (the probability that two random examples differ),
* **misclassification error** $E = 1 - \max_k p_k$.

The **gain** of a question is the impurity of the parent minus the weighted impurity of the children.

Start with the textbook example: 14 days, "play tennis?" (9 yes, 5 no).

In [ ]:
tennis = pl.DataFrame({
    "outlook": ["sunny", "sunny", "overcast", "rain", "rain", "rain", "overcast", "sunny", "sunny", "rain", "sunny", "overcast", "overcast", "rain"],
    "temperature": ["hot", "hot", "hot", "mild", "cool", "cool", "cool", "mild", "cool", "mild", "mild", "mild", "hot", "mild"],
    "humidity": ["high", "high", "high", "high", "normal", "normal", "normal", "high", "normal", "normal", "normal", "high", "normal", "high"],
    "windy": ["no", "yes", "no", "no", "no", "yes", "yes", "no", "no", "no", "yes", "yes", "no", "yes"],
    "play": ["no", "no", "yes", "yes", "yes", "no", "yes", "no", "yes", "yes", "yes", "yes", "yes", "no"],
})


def entropy(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum(axis=-1, keepdims=True)
    return -np.sum(np.where(p > 0, p * np.log2(np.where(p > 0, p, 1)), 0.0), axis=-1)


def gini(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum(axis=-1, keepdims=True)
    return 1.0 - np.sum(p**2, axis=-1)


def error(counts):
    counts = np.asarray(counts, dtype=float)
    return 1.0 - counts.max(axis=-1) / counts.sum(axis=-1)


yes = (tennis["play"] == "yes").to_numpy()
parent = np.array([(~yes).sum(), yes.sum()])
print(f"parent: {parent[1]} yes, {parent[0]} no -> entropy {entropy(parent):.3f}, Gini {gini(parent):.3f}, error {error(parent):.3f}")
rows = []
for feat in ("outlook", "temperature", "humidity", "windy"):
    res = {"entropy": entropy(parent), "gini": gini(parent), "error": error(parent)}
    for crit, fn in (("entropy", entropy), ("gini", gini), ("error", error)):
        child = 0.0
        for v in tennis[feat].unique().to_list():
            m = (tennis[feat] == v).to_numpy()
            c = np.array([(~yes[m]).sum(), yes[m].sum()])
            child += m.mean() * fn(c)
        res[crit] -= child
    rows.append((feat, res["entropy"], res["gini"], res["error"]))
table(rows, ["question: split on", "gain (entropy)", "gain (Gini)", "gain (error)"])

Outlook has the largest gain under all three criteria, so it is the root. Note that the **misclassification error gains
nothing for several questions** (a split can be useful even if no example changes class): entropy and Gini are *smooth*,
strictly concave functions of $p$, so they reward purer children even when the majority class stays the same.

In [ ]:
p = np.linspace(0.001, 0.999, 300)
two = np.c_[p, 1 - p]
plt.plot(p, entropy(two), label="entropy")
plt.plot(p, 2 * gini(two), label="Gini x 2 (rescaled)")
plt.plot(p, 2 * error(two), label="error x 2 (rescaled)")
plt.xlabel("proportion of class 1")
plt.ylabel("impurity")
plt.legend()
plt.title("impurity measures of a two-class node")
plt.show()

# a split on real data where error gains nothing but Gini does
parent = np.array([40, 40])
left, right = np.array([30, 10]), np.array([10, 30])
for name, fn in (("entropy", entropy), ("Gini", gini), ("error", error)):
    print(f"{name:<8} parent {fn(parent):.3f}, children {0.5 * fn(left) + 0.5 * fn(right):.3f}, gain {fn(parent) - 0.5 * fn(left) - 0.5 * fn(right):.3f}")
left, right = np.array([40, 10]), np.array([0, 30])
print("A split with children (40, 10) and (0, 30):")
for name, fn in (("entropy", entropy), ("Gini", gini), ("error", error)):
    print(f"{name:<8} gain {fn(parent) - (50 / 80) * fn(left) - (30 / 80) * fn(right):.3f}")

### Continuous features: the best threshold

For a numeric feature the candidate questions are "$x \le t$" with $t$ the midpoints between *consecutive sorted values*.
One pass over the sorted data with **cumulative class counts** evaluates every threshold: $O(n \log n)$ for the sort.
The function below is the core of the algorithm; the lab asks you to write it.

In [ ]:
def best_threshold(x, y, n_classes, criterion=gini):
    """Best split `x <= t` of one numeric feature: returns (threshold, gain); (None, 0) when no split helps."""
    order = np.argsort(x, kind="stable")
    xs, ys = x[order], y[order]
    n = len(ys)
    cum = np.cumsum(np.eye(n_classes)[ys], axis=0)           # class counts of the first i+1 examples
    total = cum[-1]
    left, right = cum[:-1], total - cum[:-1]
    n_left = np.arange(1, n)
    gain = criterion(total) - (n_left / n) * criterion(left) - ((n - n_left) / n) * criterion(right)
    gain = np.where(xs[:-1] < xs[1:], gain, -np.inf)          # only between different values
    i = int(np.argmax(gain))
    if not np.isfinite(gain[i]) or gain[i] <= 1e-12:
        return None, 0.0
    return float((xs[i] + xs[i + 1]) / 2), float(gain[i])


rows = []
for j in np.argsort([-best_threshold(Xb[:, j], yb, 2)[1] for j in range(Xb.shape[1])])[:6]:
    t_, g_ = best_threshold(Xb[:, j], yb, 2)
    t2, g2 = best_threshold(Xb[:, j], yb, 2, entropy)
    rows.append((names_b[j], t_, g_, g2))
table(rows, ["breast cancer feature", "best threshold (Gini)", "Gini gain", "entropy gain"])
j = names_b.index("worst_perimeter")
t_, g_ = best_threshold(Xb[:, j], yb, 2)
srt = np.sort(Xb[:, j])
print(f"worst_perimeter: {len(np.unique(srt)) - 1} candidate thresholds, the best is {t_:.1f}")
ths = (srt[:-1] + srt[1:]) / 2
plt.hist([Xb[yb == 1, j], Xb[yb == 0, j]], bins=30, label=["benign", "malignant"], stacked=True)
plt.axvline(t_, color="k", ls="--", label=f"best threshold {t_:.0f}")
plt.xlabel("worst_perimeter")
plt.legend()
plt.show()

## T2 — A tree from scratch

**The algorithm** (CART, Breiman et al. 1984): at a node with examples $S$

1. if $S$ is pure, or the depth limit or the minimum size is reached, make a **leaf** with the majority class;
2. otherwise find the (feature, threshold) with the largest gain over all features;
3. send the examples with $x_j \le t$ to the left child and the rest to the right; recurse.

It is **greedy**: each split is the best *now*, with no look-ahead.

In [ ]:
def fit_tree(X, y, n_classes, max_depth=5, min_leaf=1, criterion=gini, depth=0):
    counts = np.bincount(y, minlength=n_classes)
    node = {"counts": counts, "pred": int(counts.argmax()), "n": len(y)}
    if depth >= max_depth or counts.max() == len(y) or len(y) < 2 * min_leaf:
        return node
    best = (None, None, 0.0)
    for j in range(X.shape[1]):
        t, g = best_threshold(X[:, j], y, n_classes, criterion)
        if t is not None and g > best[2]:
            m = X[:, j] <= t
            if m.sum() >= min_leaf and (~m).sum() >= min_leaf:
                best = (j, t, g)
    if best[0] is None:
        return node
    j, t, g = best
    m = X[:, j] <= t
    node.update(feature=j, threshold=t, gain=g,
                left=fit_tree(X[m], y[m], n_classes, max_depth, min_leaf, criterion, depth + 1),
                right=fit_tree(X[~m], y[~m], n_classes, max_depth, min_leaf, criterion, depth + 1))
    return node


def predict_tree(node, X):
    out = np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        n = node
        while "feature" in n:
            n = n["left"] if x[n["feature"]] <= n["threshold"] else n["right"]
        out[i] = n["pred"]
    return out


def depth_of(node):
    return 0 if "feature" not in node else 1 + max(depth_of(node["left"]), depth_of(node["right"]))


def leaves_of(node):
    return 1 if "feature" not in node else leaves_of(node["left"]) + leaves_of(node["right"])


rows = []
for name, (X, y) in CLASSIFICATION.items():
    K = len(np.unique(y))
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    t0 = time.time()
    ours = fit_tree(a, ya, K, max_depth=6, min_leaf=1)
    t_ours = time.time() - t0
    ref = DecisionTreeClassifier(criterion="gini", max_depth=6, random_state=0).fit(a, ya)
    same = np.mean(predict_tree(ours, b) == ref.predict(b))
    rows.append((name, float(np.mean(predict_tree(ours, b) == yb_)), float(ref.score(b, yb_)), float(same), leaves_of(ours), int(ref.get_n_leaves()), t_ours))
table(rows, ["dataset, depth 6", "acc ours", "acc sklearn", "same pred.", "leaves ours", "leaves sk", "fit (s)"])

The two trees agree on (almost) every test example; the few differences come from **ties**: two features with exactly
the same gain, broken in a different order, and from the random feature order of scikit-learn.

**Reading a tree as rules.** Every leaf is a conjunction of conditions: a rule that a human can read, check and challenge.

In [ ]:
small = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xb, yb)
print(export_text(small, feature_names=names_b, decimals=2))
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(small, feature_names=names_b, class_names=["malignant", "benign"], filled=True, impurity=True, fontsize=7, ax=ax)
plt.show()

## T3 — Depth, overfitting and pruning

A tree with no limit keeps splitting until every leaf is pure: **100% training accuracy**, and a model that has memorized
the noise (Class 01). Two families of remedies:

* **pre-pruning:** stop early (`max_depth`, `min_samples_leaf`, a minimum gain);
* **post-pruning:** grow fully, then cut back the branches that do not pay for themselves: **cost-complexity pruning**
  minimizes $\ \text{error} + \alpha\cdot\#\text{leaves}$, a ridge-like penalty on the size (Class 03, $\lambda$ again).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
depths = range(1, 16)
rows = []
for ax, (name, (X, y)) in zip(axes, [(k, CLASSIFICATION[k]) for k in ("breast_cancer", "german_credit", "wine (quality >= 6)")]):
    tr_acc, cv_acc, sd = [], [], []
    for d in depths:
        m = DecisionTreeClassifier(max_depth=d, random_state=0)
        tr_acc.append(m.fit(X, y).score(X, y))
        s = cross_val_score(m, X, y, cv=StratifiedKFold(10, shuffle=True, random_state=0))
        cv_acc.append(s.mean())
        sd.append(s.std())
    ax.plot(depths, tr_acc, label="train")
    ax.errorbar(depths, cv_acc, yerr=sd, label="10-fold CV")
    ax.set(xlabel="max_depth", title=name)
    best = int(np.argmax(cv_acc))
    rows.append((name, list(depths)[best], cv_acc[best], tr_acc[-1], cv_acc[-1]))
axes[0].set_ylabel("accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()
table(rows, ["dataset", "best depth", "CV accuracy at best", "train acc, depth 15", "CV acc, depth 15"])

**Cost-complexity pruning path** (`ccp_alpha`): each increase of $\alpha$ removes the branch with the smallest
gain per leaf. The path gives the whole family of nested trees at the price of one fit.

In [ ]:
Xa, Xv, ya, yv = train_test_split(Xg, yg, test_size=0.3, stratify=yg, random_state=0)
full = DecisionTreeClassifier(random_state=0).fit(Xa, ya)
path = full.cost_complexity_pruning_path(Xa, ya)
alphas = path.ccp_alphas[::max(1, len(path.ccp_alphas) // 25)]
rows = []
for a_ in alphas:
    m = DecisionTreeClassifier(ccp_alpha=a_, random_state=0).fit(Xa, ya)
    rows.append((float(a_), int(m.get_n_leaves()), m.score(Xa, ya), m.score(Xv, yv)))
table(rows[::2], ["ccp_alpha", "leaves", "train acc", "validation acc"])
best = max(rows, key=lambda r: r[3])
print(f"full tree: {full.get_n_leaves()} leaves, train {full.score(Xa, ya):.3f}, validation {full.score(Xv, yv):.3f}")
print(f"best pruned tree: {best[1]} leaves, validation {best[3]:.3f}  (alpha {best[0]:.4f})")
plt.plot([r[1] for r in rows], [r[2] for r in rows], label="train")
plt.plot([r[1] for r in rows], [r[3] for r in rows], label="validation")
plt.xlabel("number of leaves")
plt.ylabel("accuracy")
plt.legend()
plt.title("german credit: complexity against accuracy")
plt.show()

**Other limits.** `min_samples_leaf` (a leaf needs enough examples to be trusted) is the most practical pre-pruning
knob; the table sweeps it together with the impurity criterion.

In [ ]:
rows = []
for name in ("breast_cancer", "german_credit", "heart_disease", "wine (quality >= 6)"):
    X, y = CLASSIFICATION[name]
    for crit in ("gini", "entropy"):
        for leaf in (1, 5, 20, 50):
            s = cross_val_score(DecisionTreeClassifier(criterion=crit, min_samples_leaf=leaf, random_state=0), X, y,
                                cv=StratifiedKFold(10, shuffle=True, random_state=0))
            rows.append((name, crit, leaf, s.mean(), s.std()))
df = pl.DataFrame(rows, schema=["dataset", "criterion", "min_leaf", "CV accuracy", "sd"], orient="row")
print(df.pivot(on="min_leaf", index=["dataset", "criterion"], values="CV accuracy"))

## T4 — Regression trees

For a numeric target the leaf predicts the **mean** of its examples, and a split is judged by the **reduction of the
variance** (the sum of squared errors): the same algorithm with another impurity. The prediction is a **staircase**.

In [ ]:
rng = np.random.default_rng(0)
xs = np.sort(rng.uniform(0, 6, 80))
ys = np.sin(xs) + rng.normal(0, 0.25, 80)
xg = np.linspace(-1, 7, 400)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
rows = []
for ax, d in zip(axes, (1, 2, 4, 8)):
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(xs[:, None], ys)
    ax.scatter(xs, ys, s=8)
    ax.plot(xg, m.predict(xg[:, None]), color="tab:red")
    ax.plot(xg, np.sin(xg), "k:", lw=0.8)
    ax.set_title(f"depth {d}: {m.get_n_leaves()} leaves", fontsize=9)
    rows.append((d, int(m.get_n_leaves()), float(np.sqrt(np.mean((m.predict(xs[:, None]) - ys) ** 2)))))
plt.tight_layout()
plt.show()
table(rows, ["max_depth", "leaves", "train RMSE"])
print("outside the data (x < 0 or x > 6) a tree repeats the value of its last leaf: no extrapolation; the line, however, continues the trend.")

**A split by hand.** Five points $(x, y)$: $(1,1.0), (2,1.2), (3,3.1), (4,3.0), (5,3.2)$. The best threshold is the one with the
smallest total squared error around the two means.

In [ ]:
x5 = np.array([1.0, 2, 3, 4, 5])
y5 = np.array([1.0, 1.2, 3.1, 3.0, 3.2])
print(f"no split: mean {y5.mean():.2f}, SSE {np.sum((y5 - y5.mean()) ** 2):.3f}")
rows = []
for t in (1.5, 2.5, 3.5, 4.5):
    L, R = y5[x5 <= t], y5[x5 > t]
    sse = np.sum((L - L.mean()) ** 2) + np.sum((R - R.mean()) ** 2)
    rows.append((t, float(L.mean()), float(R.mean()), float(sse)))
table(rows, ["threshold", "left mean", "right mean", "total SSE"])

**Real data, and the extrapolation failure.** Four regression sets, 5-fold RMSE of the baseline (the mean), a linear
model, and trees of increasing depth:

In [ ]:
rows = []
for name, target in (("concrete", "strength"), ("abalone", "rings"), ("auto_mpg", "mpg"), ("california_housing", "MedHouseVal")):
    X, y, _ = load_california() if name == "california_housing" else load_table(name, target)
    y = y.astype(float)
    if len(y) > 6000:
        idx = np.random.default_rng(0).choice(len(y), 6000, replace=False)
        X, y = X[idx], y[idx]
    res = {}
    for label, mk in (("mean", lambda: DecisionTreeRegressor(max_depth=0 + 1, min_samples_leaf=len(y))), ("linear", LinearRegression),
                      ("tree d=3", lambda: DecisionTreeRegressor(max_depth=3, random_state=0)),
                      ("tree d=6", lambda: DecisionTreeRegressor(max_depth=6, random_state=0)),
                      ("tree d=12", lambda: DecisionTreeRegressor(max_depth=12, random_state=0)),
                      ("tree leaf>=10", lambda: DecisionTreeRegressor(min_samples_leaf=10, random_state=0))):
        errs = []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, np.digitize(y, np.quantile(y, [0.25, 0.5, 0.75]))):
            m = mk().fit(X[tr], y[tr])
            errs.append(np.sqrt(np.mean((m.predict(X[te]) - y[te]) ** 2)))
        res[label] = float(np.mean(errs))
    rows.append((name, *res.values()))
table(rows, ["RMSE (5-fold)", "mean", "linear", "tree d=3", "tree d=6", "tree d=12", "tree leaf>=10"])

*Extrapolation:* train on the part of California housing with the lowest median income and test on the highest.
A linear model continues the trend; a tree predicts the value of the last leaf.

In [ ]:
Xh2, yh2, nh = load_california()
j = nh.index("MedInc")
cut = np.quantile(Xh2[:, j], 0.7)
tr, te = Xh2[:, j] <= cut, Xh2[:, j] > cut
rows = []
for label, m in (("mean", DecisionTreeRegressor(max_depth=1, min_samples_leaf=len(yh2))), ("linear", LinearRegression()), ("tree d=6", DecisionTreeRegressor(max_depth=6, random_state=0)),
                 ("tree leaf>=20", DecisionTreeRegressor(min_samples_leaf=20, random_state=0))):
    m.fit(Xh2[tr], yh2[tr])
    inside = np.sqrt(np.mean((m.predict(Xh2[tr]) - yh2[tr]) ** 2))
    outside = np.sqrt(np.mean((m.predict(Xh2[te]) - yh2[te]) ** 2))
    rows.append((label, float(inside), float(outside)))
table(rows, ["model", "RMSE, seen range", "RMSE, richest 30%"])

## T5 — Instability and importances

A small change of the data can change the **first split**, and then everything below it: a tree has **high variance**
(Class 01). 30 trees, each trained on a different bootstrap sample (draw $n$ examples with replacement) of the breast cancer
data:

In [ ]:
rng = np.random.default_rng(0)
Xa, Xt, ya, yt = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=0)
roots, accs, leaves = [], [], []
for b in range(30):
    idx = rng.integers(0, len(ya), len(ya))
    m = DecisionTreeClassifier(random_state=b).fit(Xa[idx], ya[idx])
    roots.append(names_b[m.tree_.feature[0]])
    accs.append(m.score(Xt, yt))
    leaves.append(m.get_n_leaves())
vals, cnt = np.unique(roots, return_counts=True)
print("feature used at the root over 30 bootstrap trees:", dict(zip(vals.tolist(), cnt.tolist())))
print(f"test accuracy of the 30 trees: mean {np.mean(accs):.3f}, sd {np.std(accs):.3f}, min {np.min(accs):.3f}, max {np.max(accs):.3f}; leaves {int(np.min(leaves))}-{int(np.max(leaves))}")
votes = np.mean([DecisionTreeClassifier(random_state=b).fit(Xa[i], ya[i]).predict(Xt) for b, i in enumerate([rng.integers(0, len(ya), len(ya)) for _ in range(30)])], axis=0)
print(f"majority vote of the 30 trees (bagging, Class 07): test accuracy {np.mean((votes > 0.5) == yt):.3f}")

**Feature importances.** A tree gives a free ranking: the total impurity decrease of the splits on each feature. But it
is **biased towards features with many distinct values** (more candidate thresholds, more chances to fit noise). Add one
*random* continuous feature and one random binary feature to the wine data and compare with the permutation importance
(the drop of the test accuracy when the feature is shuffled) on held-out data:

In [ ]:
rng = np.random.default_rng(1)
Xn = np.c_[Xw, rng.normal(size=len(yw)), rng.integers(0, 2, len(yw))]
names_n = names_w + ["RANDOM continuous", "RANDOM binary"]
Xa, Xt, ya, yt = train_test_split(Xn, yw, test_size=0.3, stratify=yw, random_state=0)
m = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(Xa, ya)
perm = permutation_importance(m, Xt, yt, n_repeats=20, random_state=0)
order = np.argsort(-m.feature_importances_)
rows = [(names_n[i], float(m.feature_importances_[i]), float(perm.importances_mean[i])) for i in order[:8]]
rows += [(n_, float(m.feature_importances_[names_n.index(n_)]), float(perm.importances_mean[names_n.index(n_)])) for n_ in ("RANDOM continuous", "RANDOM binary")]
table(rows, ["feature", "impurity importance (train)", "permutation importance (test)"])
print(f"test accuracy {m.score(Xt, yt):.3f}; the random continuous feature gets {m.feature_importances_[-2]:.3f} of the impurity importance")

**Correlated features** share the credit: duplicate the most important feature and the importance of each copy halves.

In [ ]:
j = int(np.argmax(m.feature_importances_))
Xdup = np.c_[Xn, Xn[:, j]]
a2, t2, ya2, yt2 = train_test_split(Xdup, yw, test_size=0.3, stratify=yw, random_state=0)
m2 = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(a2, ya2)
print(f"{names_n[j]}: importance {m.feature_importances_[j]:.3f} alone; with a copy: {m2.feature_importances_[j]:.3f} + {m2.feature_importances_[-1]:.3f} = {m2.feature_importances_[j] + m2.feature_importances_[-1]:.3f}")

## Summary

* A tree is a sequence of questions, grown **greedily** by the impurity gain (Gini or entropy); a numeric feature is split
  at a midpoint found by a sorted scan with cumulative counts.
* Trees overfit by construction: limit the depth or the leaf size, or prune by cost-complexity, **on validation data**.
* Regression trees predict leaf means (a staircase) and cannot extrapolate.
* Rules are readable; the *importances* and the first split are **unstable** and biased: use permutation importance and,
  next class, ensembles of trees.